# 03_v2 Usage Feature Engineering

Create leakage-controlled usage behavior features at `membership_row_id` level. This notebook does not create content features, modeling datasets for training, SHAP outputs, or trained models.


In [1]:
import csv
import json
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RAW_VIEW = PROJECT_ROOT / "_data" / "01_raw" / "View_History.csv"
STAGE02_DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "02_v2_preprocessing_policy"
MEMBERSHIP_PATH = STAGE02_DATA_DIR / "membership_v2_preprocessed.csv"
USERMAPPING_PATH = STAGE02_DATA_DIR / "usermapping_v2_policy_checked.csv"
DATA_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "data" / "03_v2_usage_feature_engineering"
TABLE_DIR = PROJECT_ROOT / "park.ingyeom" / "reports" / "tables" / "03_v2_usage_feature_engineering"
DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

WINDOWS = {"w1_3": (0, 20), "w1_4": (0, 27)}
WEEK_RANGES = {
    "week1": (0, 6),
    "week2": (7, 13),
    "week3": (14, 20),
    "week4": (21, 27),
}
BASE_FEATURES = [
    "has_watch_obs", "no_watch_obs_flag", "total_watch_time", "total_sessions", "unique_contents", "unique_watch_days",
    "avg_watch_time_per_session", "sessions_per_active_day", "active_span_days", "first_watch_rel_day", "last_watch_rel_day",
    "week1_watch_time", "week2_watch_time", "week3_watch_time", "week1_sessions", "week2_sessions", "week3_sessions",
    "week1_ratio", "week2_ratio", "week3_ratio", "w2_minus_w1_watch_time", "w3_minus_w1_watch_time",
    "max_daily_watch_time", "max_day_share", "one_minute_watch_count", "short_watch_count_le5", "short_watch_time_le5",
]
W1_4_EXTRA = ["week4_watch_time", "week4_sessions", "week4_ratio", "w4_minus_w1_watch_time", "w4_minus_w3_watch_time"]
FORBIDDEN_FEATURE_TOKENS = ["USER_KEY", "USER_NUM", "MOVIE_NUM", "reg_date", "end_date", "duration_days", "watch_date", "watch_day", "is_repurchase", "days_to_end", "days_since_last_watch_to_end"]

def snapshot(paths):
    out = {}
    for path in paths:
        out[str(path)] = {"size": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns}
    return out

def rel(path):
    return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")

def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        return reader.fieldnames or [], list(reader)

def write_csv(path, rows, fieldnames):
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, "") for field in fieldnames})

def write_json(path, payload):
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def parse_date(value, fmt):
    return datetime.strptime(value, fmt).date()

def safe_float(value):
    try:
        return float(value)
    except Exception:
        return 0.0

raw_before = snapshot([RAW_VIEW])
stage02_before = snapshot([MEMBERSHIP_PATH, USERMAPPING_PATH])

membership_cols, membership = read_csv(MEMBERSHIP_PATH)
mapping_cols, usermapping = read_csv(USERMAPPING_PATH)
view_cols, view_rows = read_csv(RAW_VIEW)

membership_by_id = {}
membership_ids_by_user_key = defaultdict(list)
for row in membership:
    mid = int(row["membership_row_id"])
    row["_mid"] = mid
    row["_reg_date"] = parse_date(row["reg_date"], "%y-%m-%d")
    row["_end_date"] = parse_date(row["end_date"], "%y-%m-%d")
    membership_by_id[mid] = row
    membership_ids_by_user_key[row["USER_KEY"]].append(mid)

user_nums_by_key = defaultdict(set)
keys_by_user_num = defaultdict(set)
for row in usermapping:
    user_nums_by_key[row["USER_KEY"]].add(row["USER_NUM"])
    keys_by_user_num[row["USER_NUM"]].add(row["USER_KEY"])

input_summary = [
    {"input_name": "membership_v2_preprocessed", "path": rel(MEMBERSHIP_PATH), "row_count": len(membership), "column_count": len(membership_cols), "role": "Stage 02 retained Membership events"},
    {"input_name": "usermapping_v2_policy_checked", "path": rel(USERMAPPING_PATH), "row_count": len(usermapping), "column_count": len(mapping_cols), "role": "USER_KEY to USER_NUM mapping for audit aggregation"},
    {"input_name": "View_History", "path": rel(RAW_VIEW), "row_count": len(view_rows), "column_count": len(view_cols), "role": "Raw view logs"},
]

# Audit-only temporary expansion from View_History to membership_row_id.
expanded_logs = []
attachment_dist = Counter()
join_expansion_counts = Counter()
temporal_counts = Counter()
temporal_by_window = {name: Counter() for name in WINDOWS}
short_counts_by_window = {name: Counter() for name in WINDOWS}

for view_index, view in enumerate(view_rows, start=1):
    user_num = view["USER_NUM"]
    user_keys = keys_by_user_num.get(user_num, set())
    attached_ids = []
    for user_key in user_keys:
        attached_ids.extend(membership_ids_by_user_key.get(user_key, []))
    attachment_dist[len(attached_ids)] += 1
    if len(attached_ids) == 0:
        join_expansion_counts["raw_view_rows_with_no_membership_attachment"] += 1
    if len(attached_ids) > 1:
        join_expansion_counts["raw_view_rows_attached_to_multiple_memberships"] += 1
    watch_date = parse_date(view["watch_day"], "%Y%m%d")
    watch_time = safe_float(view["watch_time(min)"])
    for mid in attached_ids:
        member = membership_by_id[mid]
        rel_day = (watch_date - member["_reg_date"]).days
        record = {
            "membership_row_id": mid,
            "view_row_id": view_index,
            "MOVIE_NUM": view["MOVIE_NUM"],
            "watch_time": watch_time,
            "watch_date": watch_date,
            "rel_day": rel_day,
        }
        expanded_logs.append(record)
        if watch_date < member["_reg_date"]:
            temporal_counts["watch_date_lt_reg_date"] += 1
        if watch_date == member["_end_date"]:
            temporal_counts["watch_date_eq_end_date"] += 1
        if watch_date > member["_end_date"]:
            temporal_counts["watch_date_gt_end_date"] += 1
        if rel_day > 27:
            temporal_counts["rel_day_gt_27"] += 1
        for window_name, (start, end) in WINDOWS.items():
            if watch_date >= member["_reg_date"] and start <= rel_day <= end:
                temporal_by_window[window_name]["included_logs"] += 1
                if watch_time == 1:
                    short_counts_by_window[window_name]["one_minute_watch_count"] += 1
                if watch_time <= 5:
                    short_counts_by_window[window_name]["short_watch_count_le5"] += 1
                    short_counts_by_window[window_name]["short_watch_time_le5"] += watch_time
            else:
                temporal_by_window[window_name]["excluded_logs"] += 1

join_expansion_summary = [
    {"metric": "raw_View_History_rows", "count": len(view_rows), "ratio": 1.0, "note": "Raw rows before temporary join expansion."},
    {"metric": "temporary_joined_rows", "count": len(expanded_logs), "ratio": round(len(expanded_logs)/len(view_rows), 6), "note": "Membership-event-level temporary rows used only for aggregation."},
    {"metric": "expansion_rows", "count": len(expanded_logs) - len(view_rows), "ratio": round((len(expanded_logs)-len(view_rows))/len(view_rows), 6), "note": "temporary_joined_rows - raw_View_History_rows."},
    {"metric": "raw_view_rows_attached_to_multiple_memberships", "count": join_expansion_counts["raw_view_rows_attached_to_multiple_memberships"], "ratio": round(join_expansion_counts["raw_view_rows_attached_to_multiple_memberships"]/len(view_rows), 6), "note": "Allowed temporarily, then aggregated back to membership_row_id."},
    {"metric": "raw_view_rows_with_no_membership_attachment", "count": join_expansion_counts["raw_view_rows_with_no_membership_attachment"], "ratio": round(join_expansion_counts["raw_view_rows_with_no_membership_attachment"]/len(view_rows), 6), "note": "Raw logs not attached to retained Membership rows."},
]
for attach_count, count in sorted(attachment_dist.items()):
    join_expansion_summary.append({"metric": f"attachment_count_{attach_count}", "count": count, "ratio": round(count/len(view_rows), 6), "note": "Number of membership_row_id attachments per raw view row."})

def init_stats():
    return {
        "total_watch_time": 0.0,
        "total_sessions": 0,
        "contents": set(),
        "days": set(),
        "daily_watch": Counter(),
        "first_rel": None,
        "last_rel": None,
        "week_watch": {"week1": 0.0, "week2": 0.0, "week3": 0.0, "week4": 0.0},
        "week_sessions": {"week1": 0, "week2": 0, "week3": 0, "week4": 0},
        "one_minute": 0,
        "short_count_le5": 0,
        "short_time_le5": 0.0,
    }

stats_by_window = {name: {mid: init_stats() for mid in membership_by_id} for name in WINDOWS}
for log in expanded_logs:
    mid = log["membership_row_id"]
    rel_day = log["rel_day"]
    watch_time = log["watch_time"]
    for window_name, (start, end) in WINDOWS.items():
        if start <= rel_day <= end:
            s = stats_by_window[window_name][mid]
            s["total_watch_time"] += watch_time
            s["total_sessions"] += 1
            s["contents"].add(log["MOVIE_NUM"])
            s["days"].add(rel_day)
            s["daily_watch"][rel_day] += watch_time
            s["first_rel"] = rel_day if s["first_rel"] is None else min(s["first_rel"], rel_day)
            s["last_rel"] = rel_day if s["last_rel"] is None else max(s["last_rel"], rel_day)
            for week, (w_start, w_end) in WEEK_RANGES.items():
                if w_start <= rel_day <= w_end:
                    s["week_watch"][week] += watch_time
                    s["week_sessions"][week] += 1
            if watch_time == 1:
                s["one_minute"] += 1
            if watch_time <= 5:
                s["short_count_le5"] += 1
                s["short_time_le5"] += watch_time

def build_feature_row(mid, window_name):
    s = stats_by_window[window_name][mid]
    total = s["total_watch_time"]
    sessions = s["total_sessions"]
    active_days = len(s["days"])
    max_daily = max(s["daily_watch"].values()) if s["daily_watch"] else 0.0
    row = {"membership_row_id": mid}
    def put(name, value):
        row[f"{window_name}_{name}"] = value
    put("has_watch_obs", 1 if sessions > 0 else 0)
    put("no_watch_obs_flag", 1 if sessions == 0 else 0)
    put("total_watch_time", round(total, 6))
    put("total_sessions", sessions)
    put("unique_contents", len(s["contents"]))
    put("unique_watch_days", active_days)
    put("avg_watch_time_per_session", round(total / sessions, 6) if sessions else 0)
    put("sessions_per_active_day", round(sessions / active_days, 6) if active_days else 0)
    put("active_span_days", (s["last_rel"] - s["first_rel"] + 1) if sessions else 0)
    put("first_watch_rel_day", s["first_rel"] if sessions else "")
    put("last_watch_rel_day", s["last_rel"] if sessions else "")
    for week in ["week1", "week2", "week3"]:
        put(f"{week}_watch_time", round(s["week_watch"][week], 6))
    for week in ["week1", "week2", "week3"]:
        put(f"{week}_sessions", s["week_sessions"][week])
    for week in ["week1", "week2", "week3"]:
        put(f"{week}_ratio", round(s["week_watch"][week] / total, 6) if total else 0)
    put("w2_minus_w1_watch_time", round(s["week_watch"]["week2"] - s["week_watch"]["week1"], 6))
    put("w3_minus_w1_watch_time", round(s["week_watch"]["week3"] - s["week_watch"]["week1"], 6))
    put("max_daily_watch_time", round(max_daily, 6))
    put("max_day_share", round(max_daily / total, 6) if total else 0)
    put("one_minute_watch_count", s["one_minute"])
    put("short_watch_count_le5", s["short_count_le5"])
    put("short_watch_time_le5", round(s["short_time_le5"], 6))
    if window_name == "w1_4":
        put("week4_watch_time", round(s["week_watch"]["week4"], 6))
        put("week4_sessions", s["week_sessions"]["week4"])
        put("week4_ratio", round(s["week_watch"]["week4"] / total, 6) if total else 0)
        put("w4_minus_w1_watch_time", round(s["week_watch"]["week4"] - s["week_watch"]["week1"], 6))
        put("w4_minus_w3_watch_time", round(s["week_watch"]["week4"] - s["week_watch"]["week3"], 6))
    return row

feature_rows = {window_name: [build_feature_row(mid, window_name) for mid in sorted(membership_by_id)] for window_name in WINDOWS}
fieldnames = {}
for window_name in WINDOWS:
    names = ["membership_row_id"] + [f"{window_name}_{feature}" for feature in BASE_FEATURES]
    if window_name == "w1_4":
        names += [f"{window_name}_{feature}" for feature in W1_4_EXTRA]
    fieldnames[window_name] = names
    write_csv(DATA_DIR / f"usage_features_v2_{window_name}.csv", feature_rows[window_name], names)

temporal_filter_summary = []
for metric, count in temporal_counts.items():
    temporal_filter_summary.append({"scope": "joined_membership_event_level", "metric": metric, "count": count, "note": "Audited only; not silently ignored."})
for window_name, counts in temporal_by_window.items():
    for metric, count in counts.items():
        temporal_filter_summary.append({"scope": window_name, "metric": metric, "count": count, "note": "Window inclusion uses watch_date >= reg_date and rel_day bounds."})

window_row_count_summary = []
no_watch_summary = []
short_watch_summary = []
for window_name in WINDOWS:
    rows = feature_rows[window_name]
    no_watch = sum(1 for r in rows if int(r[f"{window_name}_no_watch_obs_flag"]) == 1)
    has_watch = len(rows) - no_watch
    window_row_count_summary.append({"window": window_name, "feature_rows": len(rows), "unique_membership_row_id": len({r["membership_row_id"] for r in rows}), "expected_membership_rows": len(membership), "status": "PASS" if len(rows) == len(membership) else "FAIL"})
    no_watch_summary.append({"window": window_name, "membership_rows": len(rows), "has_watch_obs_rows": has_watch, "no_watch_obs_rows": no_watch, "no_watch_rate": round(no_watch / len(rows), 6)})
    short_watch_summary.append({"window": window_name, "one_minute_watch_count": int(short_counts_by_window[window_name]["one_minute_watch_count"]), "short_watch_count_le5": int(short_counts_by_window[window_name]["short_watch_count_le5"]), "short_watch_time_le5": round(short_counts_by_window[window_name]["short_watch_time_le5"], 6), "action": "kept_as_features_not_deleted"})

def numeric_summary(rows, window_name):
    out = []
    for col in fieldnames[window_name]:
        if col == "membership_row_id":
            continue
        vals = []
        for r in rows:
            if r[col] == "":
                continue
            try:
                vals.append(float(r[col]))
            except Exception:
                pass
        vals_sorted = sorted(vals)
        if vals_sorted:
            out.append({"window": window_name, "feature": col, "count": len(vals_sorted), "min": vals_sorted[0], "max": vals_sorted[-1], "mean": round(sum(vals_sorted)/len(vals_sorted), 6), "zero_count": sum(1 for v in vals_sorted if v == 0)})
    return out

def missing_summary(rows, window_name):
    out = []
    for col in fieldnames[window_name]:
        missing = sum(1 for r in rows if r[col] == "")
        out.append({"window": window_name, "feature": col, "missing_count": missing, "missing_rate": round(missing/len(rows), 6)})
    return out

numeric_rows = numeric_summary(feature_rows["w1_3"], "w1_3") + numeric_summary(feature_rows["w1_4"], "w1_4")
missing_rows = missing_summary(feature_rows["w1_3"], "w1_3") + missing_summary(feature_rows["w1_4"], "w1_4")

write_csv(TABLE_DIR / "03_v2_usage_input_summary.csv", input_summary, ["input_name", "path", "row_count", "column_count", "role"])
write_csv(TABLE_DIR / "03_v2_join_expansion_summary.csv", join_expansion_summary, ["metric", "count", "ratio", "note"])
write_csv(TABLE_DIR / "03_v2_temporal_filter_summary.csv", temporal_filter_summary, ["scope", "metric", "count", "note"])
write_csv(TABLE_DIR / "03_v2_window_row_count_summary.csv", window_row_count_summary, ["window", "feature_rows", "unique_membership_row_id", "expected_membership_rows", "status"])
write_csv(TABLE_DIR / "03_v2_no_watch_summary.csv", no_watch_summary, ["window", "membership_rows", "has_watch_obs_rows", "no_watch_obs_rows", "no_watch_rate"])
write_csv(TABLE_DIR / "03_v2_short_watch_summary.csv", short_watch_summary, ["window", "one_minute_watch_count", "short_watch_count_le5", "short_watch_time_le5", "action"])
write_csv(TABLE_DIR / "03_v2_usage_feature_numeric_summary.csv", numeric_rows, ["window", "feature", "count", "min", "max", "mean", "zero_count"])
write_csv(TABLE_DIR / "03_v2_usage_feature_missing_summary.csv", missing_rows, ["window", "feature", "missing_count", "missing_rate"])

summary_payload = {
    "scope": "Stage 03 usage feature engineering only.",
    "membership_rows": len(membership),
    "raw_viewhistory_rows": len(view_rows),
    "temporary_joined_rows": len(expanded_logs),
    "windows": {window: {"start_rel_day": bounds[0], "end_rel_day": bounds[1], "rows": len(feature_rows[window])} for window, bounds in WINDOWS.items()},
    "data_outputs": [rel(DATA_DIR / "usage_features_v2_w1_3.csv"), rel(DATA_DIR / "usage_features_v2_w1_4.csv"), rel(DATA_DIR / "usage_feature_summary.json")],
    "audit_outputs": [rel(TABLE_DIR / name) for name in ["03_v2_usage_input_summary.csv", "03_v2_join_expansion_summary.csv", "03_v2_temporal_filter_summary.csv", "03_v2_window_row_count_summary.csv", "03_v2_no_watch_summary.csv", "03_v2_short_watch_summary.csv", "03_v2_usage_feature_numeric_summary.csv", "03_v2_usage_feature_missing_summary.csv", "03_v2_final_checks.csv"]],
    "forbidden_feature_tokens": FORBIDDEN_FEATURE_TOKENS,
}
write_json(DATA_DIR / "usage_feature_summary.json", summary_payload)

report_path = DATA_DIR / "03_v2_usage_feature_engineering_report.md"
report_lines = [
    "# 03_v2 Usage Feature Engineering Report", "",
    "## Scope", "- Created usage behavior features only.", "- No content features, modeling dataset for training, SHAP output, or model was created.", "",
    "## Observation Windows", "- `w1_3`: rel_day 0 through 20.", "- `w1_4`: rel_day 0 through 27.", "- Features are separated by `w1_3_` and `w1_4_` prefixes.", "",
    "## Join Policy", "- `USER_KEY` and `USER_NUM` were used only for temporary joining and aggregation.", "- Expanded view logs were aggregated back to one row per `membership_row_id`.", "",
    "## Temporal Policy", "- Included logs require `watch_date >= reg_date` and rel_day inside the requested window.", "- `end_date` inclusiveness remains unresolved, so no end_date-derived features were created.", "",
    "## Row Counts", f"- w1_3 rows: {len(feature_rows['w1_3']):,}.", f"- w1_4 rows: {len(feature_rows['w1_4']):,}.", "",
    "## Output Files",
]
for path in summary_payload["data_outputs"] + summary_payload["audit_outputs"] + [rel(report_path)]:
    report_lines.append(f"- {path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

raw_after = snapshot([RAW_VIEW])
stage02_after = snapshot([MEMBERSHIP_PATH, USERMAPPING_PATH])
required_outputs = [DATA_DIR / "usage_features_v2_w1_3.csv", DATA_DIR / "usage_features_v2_w1_4.csv", DATA_DIR / "usage_feature_summary.json", report_path] + [TABLE_DIR / name for name in ["03_v2_usage_input_summary.csv", "03_v2_join_expansion_summary.csv", "03_v2_temporal_filter_summary.csv", "03_v2_window_row_count_summary.csv", "03_v2_no_watch_summary.csv", "03_v2_short_watch_summary.csv", "03_v2_usage_feature_numeric_summary.csv", "03_v2_usage_feature_missing_summary.csv"]]
w1_3_cols = set(fieldnames["w1_3"])
w1_4_cols = set(fieldnames["w1_4"])
bad_cols = []
for c in sorted(w1_3_cols | w1_4_cols):
    raw_name = c.replace("w1_3_", "").replace("w1_4_", "")
    if raw_name in FORBIDDEN_FEATURE_TOKENS or raw_name in {"watch_date", "watch_day", "raw_calendar_date"}:
        bad_cols.append(c)
end_date_derived = [c for c in sorted(w1_3_cols | w1_4_cols) if "end" in c.lower() or "days_since_last_watch_to_end" in c]
final_checks = [
    {"check": "raw_files_unchanged", "status": "PASS" if raw_before == raw_after else "FAIL", "detail": "View_History size and mtime unchanged"},
    {"check": "stage02_outputs_not_overwritten", "status": "PASS" if stage02_before == stage02_after else "FAIL", "detail": "Stage 02 membership/usermapping inputs unchanged"},
    {"check": "no_project_root_data_output_created", "status": "PASS" if not (PROJECT_ROOT / "_data" / "02_interim" / "03_v2_usage_feature_engineering").exists() else "FAIL", "detail": "Stage 03 writes only under park.ingyeom/reports"},
    {"check": "one_row_per_membership_row_id_w1_3", "status": "PASS" if len(feature_rows["w1_3"]) == len({r["membership_row_id"] for r in feature_rows["w1_3"]}) == len(membership) else "FAIL", "detail": f"rows={len(feature_rows['w1_3'])}"},
    {"check": "one_row_per_membership_row_id_w1_4", "status": "PASS" if len(feature_rows["w1_4"]) == len({r["membership_row_id"] for r in feature_rows["w1_4"]}) == len(membership) else "FAIL", "detail": f"rows={len(feature_rows['w1_4'])}"},
    {"check": "w1_3_and_w1_4_features_separated", "status": "PASS" if all(c == "membership_row_id" or c.startswith("w1_3_") for c in w1_3_cols) and all(c == "membership_row_id" or c.startswith("w1_4_") for c in w1_4_cols) else "FAIL", "detail": "window prefixes checked"},
    {"check": "no_end_date_derived_feature_created", "status": "PASS" if not end_date_derived else "FAIL", "detail": "none" if not end_date_derived else "|".join(end_date_derived)},
    {"check": "no_forbidden_identifier_or_date_feature_columns", "status": "PASS" if not bad_cols else "FAIL", "detail": "none" if not bad_cols else "|".join(bad_cols)},
    {"check": "no_content_feature_created", "status": "PASS", "detail": "Movie_Master not used; no content output created"},
    {"check": "no_model_trained", "status": "PASS", "detail": "No model training code or output"},
    {"check": "all_required_outputs_created", "status": "PASS" if all(p.exists() for p in required_outputs) else "FAIL", "detail": f"required_outputs={len(required_outputs)}"},
]
write_csv(TABLE_DIR / "03_v2_final_checks.csv", final_checks, ["check", "status", "detail"])

print("03_v2 usage feature engineering completed.")
for row in final_checks:
    print(f"{row['check']}: {row['status']} - {row['detail']}")


03_v2 usage feature engineering completed.
raw_files_unchanged: PASS - View_History size and mtime unchanged
stage02_outputs_not_overwritten: PASS - Stage 02 membership/usermapping inputs unchanged
no_project_root_data_output_created: PASS - Stage 03 writes only under park.ingyeom/reports
one_row_per_membership_row_id_w1_3: PASS - rows=23933
one_row_per_membership_row_id_w1_4: PASS - rows=23933
w1_3_and_w1_4_features_separated: PASS - window prefixes checked
no_end_date_derived_feature_created: PASS - none
no_forbidden_identifier_or_date_feature_columns: PASS - none
no_content_feature_created: PASS - Movie_Master not used; no content output created
no_model_trained: PASS - No model training code or output
all_required_outputs_created: PASS - required_outputs=12
